## Setting up InfluxDB 

First set up the packages to help in uplaoding and manipulating the data in influxdb

In [3]:
#Install operational packages to help in organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json




Ensure that Python 3 installed, this will not work with older versions of Python.
Install the influxdb3-python module. You'll use this to write and query InfluxDB. Run the command below in your terminal:

In [4]:
!pip install influxdb3-python


Install a module to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [5]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

Install more operational and system modules. To help on storing the api key safely as an environment variable and other modules that assist to uplaod data into the influxdb database

In [6]:
import os, time
from influxdb_client_3 import InfluxDBClient3, Point

Get the permission to access the database

Here, we initialize the token, organization info, and server url host that are needed to set up the initial connection to InfluxDB. The client connection is then established with the InfluxDBClient initialization.

In [7]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

token = os.environ.get("INFLUXDB_TOKEN")
org = "Air quality - Harrisburg and Philadelphia"
host = "https://us-east-1-1.aws.cloud2.influxdata.com"

client = InfluxDBClient3(host=host, token=token, org=org)

## Example of a dataset and the querrying process

Uploading the data

In [8]:
%%skip

database="pilot-harrisburg-home"


data = {
  "point1": {
    "location": "Klamath",
    "species": "bees",
    "count": 23,
  },
  "point2": {
    "location": "Portland",
    "species": "ants",
    "count": 30,
  },
  "point3": {
    "location": "Klamath",
    "species": "bees",
    "count": 28,
  },
  "point4": {
    "location": "Portland",
    "species": "ants",
    "count": 32,
  },
  "point5": {
    "location": "Klamath",
    "species": "bees",
    "count": 29,
  },
  "point6": {
    "location": "Portland",
    "species": "ants",
    "count": 40,
  },
}

for key in data:
  point = (
    Point("census")
    .tag("location", data[key]["location"])
    .field(data[key]["species"], data[key]["count"])
  )
  client.write(database=database, record=point)
  time.sleep(1) # separate points by 1 second

print("Complete. Return to the InfluxDB UI.")


### A sample querry in python 

In [23]:
query = """SELECT *
FROM 'census'
WHERE time >= now() - interval '24 hours'
AND ('bees' IS NOT NULL OR 'ants' IS NOT NULL)"""

# Execute the query
table = client.query(query=query, database="pilot-harrisburg-home", language='sql') 

# Convert to dataframe
df = table.to_pandas().sort_values(by="time")
df = pd.DataFrame(df)
df


,ants,bees,location,time


Aggregate functions take the values of all rows in a table and use them to perform an aggregate operation. The result is output as a new value in a single-row table.

In this example, we use the mean() function to calculate the average value of data points in the last 10 minutes.

In [10]:
%%skip
query = """SELECT mean(count)
FROM "census"r
WHERE time > now() - 10m"""

# Execute the query
table = client.query(query=query, database="pilot-harrisburg-home", language="influxsql")

# Convert to dataframe
df = table.to_pandas().sort_values(by="time")
print(df)


## Connecting Pilot Test Devices to the database

#### The quantaq devices


In [11]:
#Load the api key from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")

First attempt to print the account information as a test

In [12]:
%%skip
#Define the the account url as defined in the quantaq documentation 
qauntaq_aq_account_url = "https://api.quant-aq.com/v1/account/"

#Use the requests module to obtain the account information
response = requests.get(qauntaq_aq_account_url, auth=(QUANTAQ_APIKEY,""))

#Print the responses for verification
response_json_acc = response.content
response_dict = json.loads(response_json_acc)

#Print the response veritcally 
for info in response_dict.items():
    print(info)




### Automation 
Create functions that automate some of the tasks in calling and manipulating the data

In [13]:
def get_quantaq_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_json (json): a json string with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    return response_json.content

In [14]:
def upload_data_to_Influxdb(data_dict, measurement_name, influx_database_name):
    """ 
    A function that accepts data and stores it as table in a specified influxdb database

    Args:
        data_dict(dictionary):      Data to be stored in the influxdb database
        measurement_name(string):   Name of the table as it will appear in influxdb
        influx_database_name(string):    Name of the database in influxdb to store the table

    Returns:
        Confirms to have written the table to influxdb database
    """
    #Store the table name (measurement_name) and the database
    

    #Add the data to the in form of a table to the database 
    #Using influxdbs, Point module methods

    for key_parameter in data_dict:

        for key_record in data_dict[key_parameter]:
            point = (
                Point(measurement_name).field(data_dict[key_parameter], data_dict[key_parameter][key_record])

                )
            client.write(database=influx_database_name, record=point)
            time.sleep(1) #separate points by 1 second
    print (f"complete: You data is now uploaded to \
           {influx_database_name} database in influxdb")
    

Clarity automation - test

In [ ]:
def get_clarity_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://clarity-data-api.clarity.io" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """
    
    #Define the fist part of the url
    url_first_part = "https://clarity-data-api.clarity.io"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional + "/" + adds_on   
        
    complete_url = url_first_part+url_additional

    #Define the headers argument to be used to get the data
    headers_clarity = {
        'X-API-key' : CLARITY_APIKEY,
        'Accept' : "application/json"
    }

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, headers=headers_clarity)

    #Convert the file to a dictionary for easy maniputlation python
    response_dict = json.loads(response_json.content)

    return response_dict


In [15]:
data1 = {
  "point1": {
    "location": "Klamath",
    "species": "bees",
    "count": 23,
  },
  "point2": {
    "location": "Portland",
    "species": "ants",
    "count": 30,
  },
  "point3": {
    "location": "Klamath",
    "species": "bees",
    "count": 28,
  },
  "point4": {
    "location": "Portland",
    "species": "ants",
    "count": 32,
  },
  "point5": {
    "location": "Klamath",
    "species": "bees",
    "count": 29,
  },
  "point6": {
    "location": "Portland",
    "species": "ants",
    "count": 40,
  },
}

In [16]:
test_upload1 = upload_data_to_Influxdb(data1, "census2", "pilot-harrisburg-home")

TypeError: unhashable type: 'dict'

In [ ]:
list_keys = []
for key1 in data1:
    list_keys.append(key1)
list_keys   
for key in list_keys:
    for key2 in data1[key]:
        print(f"{key}: {key2}")


point1: location
point1: species
point1: count
point2: location
point2: species
point2: count
point3: location
point3: species
point3: count
point4: location
point4: species
point4: count
point5: location
point5: species
point5: count
point6: location
point6: species
point6: count


In [ ]:
print(data1['point1']['location'])

Klamath


In [ ]:
%%skip

database="pilot-harrisburg-home"


data = {
  "point1": {
    "location": "Klamath",
    "species": "bees",
    "count": 23,
  },
  "point2": {
    "location": "Portland",
    "species": "ants",
    "count": 30,
  },
  "point3": {
    "location": "Klamath",
    "species": "bees",
    "count": 28,
  },
  "point4": {
    "location": "Portland",
    "species": "ants",
    "count": 32,
  },
  "point5": {
    "location": "Klamath",
    "species": "bees",
    "count": 29,
  },
  "point6": {
    "location": "Portland",
    "species": "ants",
    "count": 40,
  },
}

for key in data:
  point = (
    Point("census")
    .tag("location", data[key]["location"])
    .field(data[key]["species"], data[key]["count"])
  )
  client.write(database=database, record=point)
  time.sleep(1) # separate points by 1 second

print("Complete. Return to the InfluxDB UI.")


In [ ]:
binary_data = get_quantaq_data(['v1', 'devices', 'MOD-X-00959', 'data', '?page=1917&per_page=50'])
json_data = binary_data.decode("utf-8") 
dict_data = json.loads(json_data)
dict_data['meta']
df_test = pd.DataFrame(dict_data['data'])
dict_data['data']

[{'co': None,
  'geo': {'lat': None, 'lon': None},
  'met': {'wx_u': 0.0,
   'wx_v': 0.0,
   'wx_wd': 0.0,
   'wx_ws': 0.0,
   'wx_ws_scalar': 0.0},
  'model': {'gas': {'co': None, 'no': None, 'no2': None, 'o3': None},
   'pm': {'pm1': None, 'pm10': None, 'pm25': None}},
  'no': None,
  'no2': None,
  'o3': None,
  'pm1': None,
  'pm10': None,
  'pm25': None,
  'raw_data_id': 2844611,
  'sn': 'MOD-X-00959',
  'timestamp': '2025-06-17T15:51:27',
  'timestamp_local': '2025-06-17T15:51:27',
  'url': 'https://api.quant-aq.com/v1/devices/MOD-X-00959/data/2844611'},
 {'co': None,
  'geo': {'lat': None, 'lon': None},
  'met': {'wx_u': 0.0,
   'wx_v': 0.0,
   'wx_wd': 0.0,
   'wx_ws': 0.0,
   'wx_ws_scalar': 0.0},
  'model': {'gas': {'co': None, 'no': None, 'no2': None, 'o3': None},
   'pm': {'pm1': None, 'pm10': None, 'pm25': None}},
  'no': None,
  'no2': None,
  'o3': None,
  'pm1': None,
  'pm10': None,
  'pm25': None,
  'raw_data_id': 2844610,
  'sn': 'MOD-X-00959',
  'timestamp': '2025-0

In [17]:
%pwd

'c:\\Users\\Owner\\Documents\\GitHub\\air-quality\\harrisburg-philly-aq'

In [ ]:
#Test by wrting a downladed csv file converted to a data frame
quant_csv_data_sample = pd.read_csv("data-from-monitors/quantaq-hourly-MOD-X-00959-HOUR-11-09-to-11-15-2025.csv")
quant_data_df = pd.DataFrame(quant_csv_data_sample)
#Write it to the database
client.write(record=quant_data_df, )

SyntaxError: expected argument value expression (3921497145.py, line 5)